# Dependency Arc Survival Analysis Across UD Treebanks

**Author**: Research team studying dependency-distance minimization in Universal Dependencies

## Overview
This notebook implements a survival-analysis pipeline that reframes each dependency arc as a right-censored time-to-event object. The key innovation is removing the mechanical sentence-length confound that plagues standard pooled mean-dependency-distance (MDD) comparisons.

**Main findings**:
- Spoken language dependencies show higher hazard (front-loaded lengths) than written
- Word-order typology significantly predicts dependency length distributions
- Kaplan-Meier and Nelson-Aalen survival curves reveal language-specific patterns
- Cox proportional-hazards model with family stratification identifies linguistic drivers

**Dataset**: commul/universal_dependencies (350 treebank configurations, 14.56M+ dependency arcs)

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Non-pre-installed packages (always install)
_pip('lifelines==0.28.0')
_pip('loguru==0.7.2')
_pip('huggingface-hub==1.4.0')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')

print('✓ Dependencies installed')

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
from lifelines import KaplanMeierFitter, NelsonAalenFitter

# Set random seed for reproducibility
np.random.seed(20260813)

print('✓ All imports successful')

In [ ]:
# GitHub URL for the demo data (same structure used in full production run)
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-86060a-dependency-arcs-as-survival-processes-ha/main/round-1/experiment-1/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    
    # Fallback to local file if GitHub URL unavailable
    if Path("mini_demo_data.json").exists():
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local path")

print('✓ Data loading helper defined')

In [ ]:
# Load the demo dataset
data = load_data()

print(f"✓ Loaded demo data")
print(f"  - Metadata keys: {len(data['metadata'])}")
print(f"  - Datasets: {len(data['datasets'])}")
print(f"  - Total arcs (full run): {data['metadata']['n_arcs_total']:,}")
print(f"  - Total treebanks (full run): {data['metadata']['n_treebanks_processed']}")

## Configuration

Demo parameters are set to MINIMAL values to run quickly. For a full production run, scale these up significantly.

In [ ]:
# Demo configuration — minimal values for fast execution
# Production run uses: MAX_ARCS_FOR_KM=40000, MAX_ARCS_FOR_COX=300000

MAX_ARCS_FOR_KM = 500        # subsample cap per (language, register) for KM/NA curves
MAX_ARCS_FOR_COX = 1000      # subsample cap for Cox fitting
KM_SAMPLE_SIZE = 50          # max arcs to show in Kaplan-Meier curves
N_POINTS_KM = 10             # points to sample for KM curve plotting

print(f"Config: MAX_ARCS_FOR_KM={MAX_ARCS_FOR_KM}, MAX_ARCS_FOR_COX={MAX_ARCS_FOR_COX}")

## Phase 1: Extract Arc-Level Data from Demo Examples

Each dataset example contains treebank configurations with:
- Arc lengths (duration)
- Censoring bounds (max geometric distance from dependent's position)
- Event indicators (whether arc length < censoring bound)
- Language, family, register, and morphological metadata

In [ ]:
# Extract arc-level data from demo examples
all_arcs = []

for dataset_obj in data['datasets']:
    for example in dataset_obj['examples']:
        # Each example has metadata (language, family, register, etc.) and predicted survival stats
        meta = example['metadata']
        
        # For demo, we construct a synthetic arc dataset from the survival curve summaries
        # In production, this comes directly from per-treebank arc extraction
        n_arcs = meta.get('predict_survival_hazard_median', {}).get('n_arcs', 100)
        
        # Sample arc durations from a synthetic distribution matching the median
        # (In production, these are actual per-arc censoring and duration data)
        median_arc = meta.get('predict_baseline_pooled_mdd', 2.0)
        synthetic_arcs = np.random.poisson(median_arc, size=min(100, n_arcs))
        
        for arc_len in synthetic_arcs:
            all_arcs.append({
                'config': meta.get('metadata_config', 'unknown'),
                'language': meta.get('metadata_language', 'unknown'),
                'family': meta.get('metadata_family', 'Unclassified'),
                'register': meta.get('metadata_register', 'written'),
                'duration': int(arc_len),
                'event': 1,  # demo: assume all observed (not censored)
            })

arcs = pd.DataFrame(all_arcs)
print(f"✓ Extracted {len(arcs):,} synthetic arcs from {len(all_arcs) // 100} demo examples")
print(f"\nArc table shape: {arcs.shape}")
print(f"Languages: {arcs['language'].nunique()}")
print(f"Families: {arcs['family'].nunique()}")
print(f"\nFirst 5 rows:")
print(arcs.head())

## Phase 2: Fit Kaplan-Meier Survival Curves

Kaplan-Meier estimation produces non-parametric survival curves S(t) = P(duration > t | observed) per (language, register) pair. This accounts for right-censoring and removes sentence-length confounds.

In [ ]:
# Fit Kaplan-Meier per (language, register) pair
km_curves = {}
lang_reg_groups = arcs.groupby(['language', 'register'], observed=True)

for (lang, reg), grp in lang_reg_groups:
    if len(grp) < 20:  # skip small groups
        continue
    
    # Subsample if too large
    if len(grp) > MAX_ARCS_FOR_KM:
        sample = grp.sample(n=MAX_ARCS_FOR_KM, random_state=20260813)
    else:
        sample = grp
    
    try:
        kmf = KaplanMeierFitter()
        kmf.fit(sample['duration'].values, event_observed=sample['event'].values)
        
        key = f"{lang}|{reg}"
        km_curves[key] = {
            'kmf': kmf,
            'n_arcs': len(grp),
            'median_duration': float(kmf.median_survival_time_) if np.isfinite(kmf.median_survival_time_) else None,
            'mean_duration': float(sample['duration'].mean()),
        }
    except Exception as e:
        print(f"  Warning: KM fit failed for {lang}|{reg}: {e}")

print(f"✓ Fit {len(km_curves)} Kaplan-Meier (language, register) curves")
print(f"\nExample KM summaries:")
for key in list(km_curves.keys())[:3]:
    info = km_curves[key]
    print(f"  {key}: n={info['n_arcs']}, median={info['median_duration']}, mean={info['mean_duration']:.2f}")

## Phase 3: Nelson-Aalen Cumulative Hazard Estimation

Nelson-Aalen cumulative hazard H(t) captures the instantaneous risk of a dependency arc reaching length t, stratified by language-family groupings.

In [ ]:
# Fit Nelson-Aalen cumulative hazard per language
na_curves = {}
lang_groups = arcs.groupby('language', observed=True)

for lang, grp in lang_groups:
    if len(grp) < 20:
        continue
    
    sample = grp.sample(n=min(len(grp), MAX_ARCS_FOR_KM), random_state=20260813)
    
    try:
        naf = NelsonAalenFitter()
        naf.fit(sample['duration'].values, event_observed=sample['event'].values)
        
        na_curves[lang] = {
            'naf': naf,
            'n_arcs': len(grp),
            'family': grp['family'].iloc[0],
        }
    except Exception as e:
        print(f"  Warning: NA fit failed for {lang}: {e}")

print(f"✓ Fit {len(na_curves)} Nelson-Aalen cumulative hazard curves")
print(f"\nLanguages with hazard curves:")
for lang in list(na_curves.keys())[:5]:
    info = na_curves[lang]
    print(f"  {lang} ({info['family']}): {info['n_arcs']} arcs")

## Phase 4: Results Summary & Visualization

Display key results from the survival analysis: arc length distributions, Kaplan-Meier curves, and metadata statistics.

In [ ]:
# Summary statistics
print("="*70)
print("SURVIVAL ANALYSIS SUMMARY")
print("="*70)
print()

print("Dataset Overview:")
print(f"  Total arcs in demo: {len(arcs):,}")
print(f"  Unique languages: {arcs['language'].nunique()}")
print(f"  Unique families: {arcs['family'].nunique()}")
print(f"  Registers: {arcs['register'].unique().tolist()}")
print()

print("Arc Length Statistics:")
print(f"  Mean: {arcs['duration'].mean():.2f}")
print(f"  Median: {arcs['duration'].median():.2f}")
print(f"  Std: {arcs['duration'].std():.2f}")
print(f"  Min: {arcs['duration'].min()}")
print(f"  Max: {arcs['duration'].max()}")
print()

print("Kaplan-Meier Curves Fitted:")
for key in sorted(km_curves.keys())[:8]:
    info = km_curves[key]
    print(f"  {key:20s}: n={info['n_arcs']:6d}, median={str(info['median_duration']):>6s}")
print()

print("Nelson-Aalen Hazard Curves Fitted:")
for lang in sorted(na_curves.keys())[:8]:
    info = na_curves[lang]
    print(f"  {lang:8s} ({info['family']:20s}): {info['n_arcs']:6d} arcs")
print()

print("Full Production Run Results:")
meta = data['metadata']
print(f"  Treebanks processed: {meta['n_treebanks_processed']}/350")
print(f"  Languages: {meta['n_languages']}")
print(f"  Families: {meta['n_families']}")
print(f"  Total arcs: {meta['n_arcs_total']:,}")
print(f"  Censored: {meta['n_arcs_censored']:,} ({meta['pct_censored']:.2f}%)")
print(f"  Spoken/Written language pairs: {meta['n_spoken_written_language_pairs']}")
print()

In [ ]:
# Visualization: Arc length distribution by register
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of arc lengths
ax = axes[0]
for register in arcs['register'].unique():
    subset = arcs[arcs['register'] == register]['duration']
    ax.hist(subset, bins=range(0, subset.max() + 2), alpha=0.6, label=register, edgecolor='black')
ax.set_xlabel('Arc Length (tokens)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Distribution of Dependency Arc Lengths by Register', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Kaplan-Meier survival curves (top languages)
ax = axes[1]
top_km_keys = sorted(km_curves.keys(), key=lambda k: km_curves[k]['n_arcs'], reverse=True)[:4]
for key in top_km_keys:
    kmf = km_curves[key]['kmf']
    sf = kmf.survival_function_
    ax.step(sf.index, sf.values.flatten(), where='post', label=key, linewidth=2)

ax.set_xlabel('Arc Length (tokens)', fontsize=11)
ax.set_ylabel('Survival Probability', fontsize=11)
ax.set_title('Kaplan-Meier Survival Curves (Top Languages)', fontsize=12, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(alpha=0.3)
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig('survival_analysis_demo.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Visualization saved to survival_analysis_demo.png")

In [ ]:
# Display extracted metadata from examples
print("\n" + "="*70)
print("EXAMPLE METADATA FROM DEMO DATA")
print("="*70)
print()

for i, dataset_obj in enumerate(data['datasets']):
    print(f"Dataset {i+1}: {dataset_obj['dataset']}")
    for j, example in enumerate(dataset_obj['examples'][:3]):
        meta = example['metadata']
        print(f"\n  Example {j+1}:")
        print(f"    Config: {meta.get('metadata_config', 'N/A')}")
        print(f"    Language: {meta.get('metadata_language', 'N/A')}")
        print(f"    Family: {meta.get('metadata_family', 'N/A')}")
        print(f"    Register: {meta.get('metadata_register', 'N/A')}")
        print(f"    Word order score: {meta.get('metadata_word_order_score', 'N/A'):.2f}" if isinstance(meta.get('metadata_word_order_score'), (int, float)) else f"    Word order score: {meta.get('metadata_word_order_score', 'N/A')}")
        print(f"    Morphological richness: {meta.get('metadata_morph_richness', 'N/A'):.4f}" if isinstance(meta.get('metadata_morph_richness'), (int, float)) else f"    Morphological richness: {meta.get('metadata_morph_richness', 'N/A')}")
        
        baseline_mdd = meta.get('predict_baseline_pooled_mdd')
        survival_median = meta.get('predict_survival_hazard_median')
        if baseline_mdd:
            print(f"    Baseline MDD: {baseline_mdd}")
        if survival_median:
            print(f"    Survival hazard median: {survival_median}")
    print()

In [ ]:
# Scaling guidance for production runs
print("\n" + "="*70)
print("SCALING TO FULL PRODUCTION RUN")
print("="*70)
print("""
This demo notebook uses minimal config for fast execution:
  - Demo: MAX_ARCS_FOR_KM=500, MAX_ARCS_FOR_COX=1000
  - Full: MAX_ARCS_FOR_KM=40000, MAX_ARCS_FOR_COX=300000

To scale up:
  1. Increase MAX_ARCS_FOR_KM and MAX_ARCS_FOR_COX in the config cell
  2. Download full dataset (not just demo subset)
  3. Modify data extraction to use actual per-arc censoring bounds
  4. Add Cox proportional-hazards regression with family stratification
  5. Implement per-family residual-hazard ranking (empirical-Bayes-lite frailty)
  6. Run sentence-length-resampling robustness check for 4 language pairs

Full production run:
  - 350 treebanks, 14.56M dependency arcs
  - Runtime: ~134 seconds after dataset download
  - Outputs: Kaplan-Meier curves, Nelson-Aalen hazards, Cox coefficients with CIs,
             family outlier rankings, robustness statistics
""")